# ECG-FM Stage 2 Fine-tuning
**ECG-FM foundation model (90M params) — full encoder fine-tune on PTB-XL**

### Before running:
1. In Colab menu: **Runtime → Change runtime type → T4 GPU**
2. Upload these two files to your Google Drive under `My Drive/EKG_models/`:
   - `mimic_iv_ecg_physionet_pretrained.pt` (~365 MB) — from your local `models/ecgfm/`
   - `ecgfm_stage1.pt` (~365 MB) — from your local `models/`

Then run all cells in order.

In [ ]:
# Cell 1 — Check GPU and RAM
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import psutil
print('System RAM:', round(psutil.virtual_memory().total / 1e9, 1), 'GB')
print('RAM available:', round(psutil.virtual_memory().available / 1e9, 1), 'GB')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_DIR = '/content/drive/MyDrive/EKG_models'
assert os.path.exists(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt'), \
    'Missing: upload mimic_iv_ecg_physionet_pretrained.pt to Google Drive/EKG_models/'
assert os.path.exists(f'{MODEL_DIR}/ecgfm_stage1.pt'), \
    'Missing: upload ecgfm_stage1.pt to Google Drive/EKG_models/'
print('Drive files found OK')

In [ ]:
# Cell 3 — Clone repo and install dependencies
!git clone https://github.com/shlomih/EKG.git
%cd EKG
!pip install -q wfdb scipy scikit-learn
print('Setup done')

In [ ]:
# Cell 4 — Download PTB-XL dataset (~4 GB, takes ~10 min)
# This downloads directly from PhysioNet — faster than uploading from your PC
import os
os.makedirs('ekg_datasets/ptbxl', exist_ok=True)

if not os.path.exists('ekg_datasets/ptbxl/ptbxl_database.csv'):
    print('Downloading PTB-XL (this takes ~10 minutes)...')
    import wfdb
    wfdb.dl_database('ptb-xl', 'ekg_datasets/ptbxl')
    print('PTB-XL download complete')
else:
    print('PTB-XL already present, skipping download')

# Verify
import pandas as pd
df = pd.read_csv('ekg_datasets/ptbxl/ptbxl_database.csv')
print(f'PTB-XL records: {len(df)}')

In [ ]:
# Cell 5 — Copy model files from Google Drive into the project
import shutil, os

os.makedirs('models/ecgfm', exist_ok=True)

shutil.copy(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt',
            'models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_DIR}/ecgfm_stage1.pt',
            'models/ecgfm_stage1.pt')

print('Model files copied:')
print(' ', round(os.path.getsize('models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')/1e6), 'MB — pretrained encoder')
print(' ', round(os.path.getsize('models/ecgfm_stage1.pt')/1e6), 'MB — stage 1 checkpoint')

In [ ]:
# Cell 6 — Run Stage 2 fine-tuning
# Full encoder unfreeze on GPU (~90M trainable params)
# Expected: ~20 min/epoch, ~3 hours total with early stopping
# Checkpoint saved automatically at every improvement

!python -u ecgfm_finetune.py --stage 2

In [ ]:
# Cell 7 — Save result back to Google Drive
import os, shutil

if os.path.exists('models/ecgfm_stage2.pt'):
    shutil.copy('models/ecgfm_stage2.pt', f'{MODEL_DIR}/ecgfm_stage2.pt')
    size_mb = round(os.path.getsize(f'{MODEL_DIR}/ecgfm_stage2.pt') / 1e6)
    print(f'Saved ecgfm_stage2.pt ({size_mb} MB) to Google Drive')

    # Print final metrics from checkpoint
    import torch
    ckpt = torch.load('models/ecgfm_stage2.pt', map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print(f"\nFinal test results:")
    print(f"  Accuracy  : {tm.get('acc', 0):.1%}")
    print(f"  HYP F1    : {tm.get('hyp_f1', 0):.3f}")
    print(f"  Macro F1  : {tm.get('macro_f1', 0):.3f}")
else:
    print('WARNING: ecgfm_stage2.pt not found — training may not have completed')

## After training

Download `ecgfm_stage2.pt` from your Google Drive (`EKG_models/ecgfm_stage2.pt`) to your local `models/` folder.

### What to expect
| Metric | Stage 1 baseline | Target |
|---|---|---|  
| HYP F1 | 0.375 test | > 0.50 |
| Macro F1 | 0.492 test | > 0.70 |

If ECG-FM Stage 2 beats the current multi-label CNN (MacroF1=0.699, MacroAUROC=0.972), it can replace the CNN backbone.

See `DECISIONS.md` → D-006 and D-007 for context.